# Visual-Time-Series HHSA Examples: Ensemble Sift, Masked Sift, CEEMDAN, and ICEEMDAN

This notebook demonstrates visual data processing by extracting a one-dimensional luminance trace from a video or image stack, then applying HHSA with the four supported decomposition methods.


In [ ]:
# Make the repository package importable when this notebook is opened from examples/.
from pathlib import Path
import sys
import warnings

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "examples":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal as scipy_signal
from scipy.signal import detrend, resample_poly, welch

from hhsa import mode_energy, run_hhsa

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})


## Load Visual Data and Extract an ROI Trace

HHSA currently analyzes one-dimensional signals. For visual data, this notebook converts frames into an ROI luminance time series. Set `VIDEO_PATH` or `IMAGE_STACK_PATH` for your own visual data, or use the synthetic stack fallback.


In [ ]:
VIDEO_PATH = None       # Example: Path("/absolute/path/to/movie.mp4")
IMAGE_STACK_PATH = None # Example: Path("/absolute/path/to/frames.npy") with shape frames x height x width


def make_synthetic_visual_stack(seconds=8.0, frame_rate=60.0, height=72, width=96, random_state=13):
    rng = np.random.default_rng(random_state)
    n_frames = int(seconds * frame_rate)
    y, x = np.mgrid[0:height, 0:width]
    frames = np.empty((n_frames, height, width), dtype=float)
    trace = np.empty(n_frames, dtype=float)

    for idx in range(n_frames):
        t = idx / frame_rate
        flicker = 0.18 * np.sin(2 * np.pi * 8.0 * t)
        slow = 0.12 * np.sin(2 * np.pi * 0.7 * t)
        motion_phase = 2 * np.pi * (x / 24.0 + 2.0 * t)
        grating = 0.20 * np.sin(motion_phase)
        transient = 0.35 * np.exp(-0.5 * ((t - seconds * 0.55) / 0.35) ** 2)
        blob = transient * np.exp(-(((x - width * 0.56) ** 2) / (2 * 13 ** 2) + ((y - height * 0.48) ** 2) / (2 * 10 ** 2)))
        frame = 0.5 + slow + flicker + grating + blob + 0.04 * rng.normal(size=(height, width))
        frames[idx] = np.clip(frame, 0, 1)
        roi = frames[idx, height // 3 : 2 * height // 3, width // 3 : 2 * width // 3]
        trace[idx] = roi.mean()
    return frames, zscore(detrend(trace)), frame_rate, "synthetic visual luminance stack fallback"


def luminance_from_rgb(frame):
    arr = np.asarray(frame, dtype=float)
    if arr.ndim == 2:
        return arr
    if arr.shape[-1] >= 3:
        return 0.2126 * arr[..., 0] + 0.7152 * arr[..., 1] + 0.0722 * arr[..., 2]
    return arr[..., 0]


def load_visual_trace(video_path=None, image_stack_path=None, seconds=8.0, target_frame_rate=60.0):
    if image_stack_path is not None and Path(image_stack_path).exists():
        frames = np.load(image_stack_path)
        frames = np.asarray([luminance_from_rgb(frame) for frame in frames], dtype=float)
        n = min(frames.shape[0], int(seconds * target_frame_rate))
        frames = frames[:n]
        h, w = frames.shape[1:3]
        roi = frames[:, h // 3 : 2 * h // 3, w // 3 : 2 * w // 3]
        trace = roi.mean(axis=(1, 2))
        return frames, zscore(detrend(trace)), float(target_frame_rate), f"image stack: {image_stack_path}"

    if video_path is not None and Path(video_path).exists():
        try:
            import imageio.v3 as iio

            meta = iio.immeta(video_path)
            frame_rate = float(meta.get("fps", target_frame_rate))
            max_frames = int(seconds * frame_rate)
            frames = []
            for idx, frame in enumerate(iio.imiter(video_path)):
                if idx >= max_frames:
                    break
                frames.append(luminance_from_rgb(frame))
            frames = np.asarray(frames, dtype=float)
            h, w = frames.shape[1:3]
            roi = frames[:, h // 3 : 2 * h // 3, w // 3 : 2 * w // 3]
            trace = roi.mean(axis=(1, 2))
            return frames, zscore(detrend(trace)), frame_rate, f"video ROI luminance: {video_path}"
        except Exception as exc:
            print(f"Video load failed, using synthetic stack instead: {type(exc).__name__}: {exc}")

    return make_synthetic_visual_stack(seconds=seconds, frame_rate=target_frame_rate, random_state=42)


frames, signal, sample_rate, source = load_visual_trace(VIDEO_PATH, IMAGE_STACK_PATH, seconds=8.0, target_frame_rate=60.0)
time = np.arange(signal.size) / sample_rate
print(source)
print(f"frames={frames.shape[0]}, frame_rate={sample_rate:.1f} Hz, duration={signal.size / sample_rate:.2f} s, frame_shape={frames.shape[1:]}")


## Input Quality Checks

These plots verify that the extracted one-dimensional signal is centered, finite, scaled reasonably, and has a plausible frequency profile before decomposition.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
frame_indices = np.linspace(0, frames.shape[0] - 1, 4, dtype=int)
for ax, idx in zip(axes, frame_indices):
    ax.imshow(frames[idx], cmap="gray", vmin=np.nanpercentile(frames, 1), vmax=np.nanpercentile(frames, 99))
    ax.set_title(f"frame {idx}")
    ax.axis("off")
fig.suptitle("Visual frames used to extract ROI luminance")
fig.tight_layout()

plot_input_qc(signal, sample_rate, title=f"Input visual ROI trace QC: {source}", spectrogram=True, max_frequency=25.0);


## Shared Diagnostic Plotting Helpers

The same helper functions are used for every decomposition method: first-layer IMFs, reconstruction, AM/FM tracks, second-layer amplitude modulation, spectra, and cross-method summaries.


In [ ]:
def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + np.finfo(float).eps)


def safe_upper_frequency(sample_rate, requested):
    return min(float(requested), 0.45 * float(sample_rate))


def plot_input_qc(x, sample_rate, title, *, spectrogram=False, max_frequency=None):
    t = np.arange(x.size) / sample_rate
    max_frequency = safe_upper_frequency(sample_rate, max_frequency or sample_rate / 2)
    freqs, psd = welch(x, fs=sample_rate, nperseg=min(512, x.size))

    rows = 4 if spectrogram else 3
    fig, axes = plt.subplots(rows, 1, figsize=(11, 2.35 * rows))
    axes[0].plot(t, x, color="black", linewidth=0.9)
    axes[0].set_title(title)
    axes[0].set_ylabel("amplitude")

    zoom_n = min(x.size, int(2.0 * sample_rate))
    axes[1].plot(t[:zoom_n], x[:zoom_n], color="tab:blue", linewidth=0.9)
    axes[1].set_title("Zoomed segment")
    axes[1].set_ylabel("amplitude")

    axes[2].semilogy(freqs, psd, color="tab:purple", linewidth=1.0)
    axes[2].set_xlim(0, max_frequency)
    axes[2].set_xlabel("Frequency (Hz)")
    axes[2].set_ylabel("PSD")
    axes[2].set_title("Welch PSD")

    if spectrogram:
        f, tt, Sxx = scipy_signal.spectrogram(x, fs=sample_rate, nperseg=min(256, x.size), noverlap=min(128, max(0, x.size // 4)))
        keep = f <= max_frequency
        im = axes[3].pcolormesh(tt, f[keep], 10 * np.log10(Sxx[keep] + np.finfo(float).eps), shading="auto", cmap="magma")
        axes[3].set_xlabel("Time (s)")
        axes[3].set_ylabel("Frequency (Hz)")
        axes[3].set_title("Spectrogram")
        fig.colorbar(im, ax=axes[3], label="dB")

    fig.tight_layout()
    return fig, axes


def plot_decomposition_diagnostics(result, label, max_modes=5):
    n_modes = min(result.imfs.shape[0], max_modes)
    rows = n_modes + 3
    t = np.arange(result.signal.size) / result.sample_rate

    fig, axes = plt.subplots(rows, 1, sharex=True, figsize=(11, max(6, 1.35 * rows)))
    axes[0].plot(t, result.signal, color="black", linewidth=0.9)
    axes[0].set_title(f"{label}: signal, IMFs, residue")
    axes[0].set_ylabel("signal")

    for idx in range(n_modes):
        axes[idx + 1].plot(t, result.imfs[idx], linewidth=0.9)
        axes[idx + 1].set_ylabel(f"IMF {idx + 1}")

    axes[n_modes + 1].plot(t, result.residue, color="tab:red", linewidth=0.9)
    axes[n_modes + 1].set_ylabel("residue")

    axes[-1].plot(t, result.signal, color="black", linewidth=1.0, label="signal")
    axes[-1].plot(t, result.reconstruction, color="tab:green", linewidth=0.9, alpha=0.85, label="IMFs + residue")
    axes[-1].set_ylabel("recon")
    axes[-1].set_xlabel("Time (s)")
    axes[-1].legend(loc="upper right")
    axes[-1].set_title(f"relative reconstruction error = {result.reconstruction_error:.3e}")

    fig.tight_layout()
    return fig, axes


def plot_am_fm_diagnostics(result, label, max_modes=4, max_frequency=80.0):
    n_modes = min(result.imfs.shape[0], max_modes)
    if n_modes == 0:
        print(f"{label}: no IMFs were extracted.")
        return None, None

    t = np.arange(result.signal.size) / result.sample_rate
    max_frequency = safe_upper_frequency(result.sample_rate, max_frequency)
    fig, axes = plt.subplots(n_modes, 2, sharex=True, figsize=(12, max(5, 1.9 * n_modes)))
    if n_modes == 1:
        axes = np.asarray([axes])

    for idx in range(n_modes):
        axes[idx, 0].plot(t, result.amplitude[idx], color="tab:blue", linewidth=0.9)
        axes[idx, 0].set_ylabel(f"IMF {idx + 1}")
        axes[idx, 0].set_title("Instantaneous amplitude" if idx == 0 else "")

        freq = np.clip(result.frequency[idx], 0, max_frequency)
        axes[idx, 1].plot(t, freq, color="tab:orange", linewidth=0.9)
        axes[idx, 1].set_ylim(0, max_frequency)
        axes[idx, 1].set_title("Instantaneous frequency" if idx == 0 else "")

    axes[-1, 0].set_xlabel("Time (s)")
    axes[-1, 1].set_xlabel("Time (s)")
    fig.suptitle(f"{label}: AM/FM checks", y=1.01)
    fig.tight_layout()
    return fig, axes


def strongest_imf_index(result):
    if result.imfs.size == 0:
        return None
    return int(np.argmax(mode_energy(result.imfs)))


def plot_second_layer_diagnostics(result, label, carrier_index=None, max_am_modes=3, max_frequency=40.0):
    if carrier_index is None:
        carrier_index = strongest_imf_index(result)
    if carrier_index is None or carrier_index >= len(result.am_imfs):
        print(f"{label}: no second-layer AM decomposition is available.")
        return None, None

    am_modes = result.am_imfs[carrier_index]
    am_residue = result.am_residues[carrier_index]
    envelope = result.amplitude[carrier_index]
    n_modes = min(am_modes.shape[0], max_am_modes)
    t = np.arange(result.signal.size) / result.sample_rate

    fig, axes = plt.subplots(n_modes + 2, 1, sharex=True, figsize=(11, max(5, 1.4 * (n_modes + 2))))
    axes[0].plot(t, envelope, color="tab:blue", linewidth=0.9)
    axes[0].set_title(f"{label}: second-layer AM decomposition from carrier IMF {carrier_index + 1}")
    axes[0].set_ylabel("envelope")

    for idx in range(n_modes):
        axes[idx + 1].plot(t, am_modes[idx], linewidth=0.9)
        axes[idx + 1].set_ylabel(f"AM IMF {idx + 1}")

    axes[-1].plot(t, am_residue, color="tab:red", linewidth=0.9)
    axes[-1].set_ylabel("AM residue")
    axes[-1].set_xlabel("Time (s)")
    fig.tight_layout()

    if result.am_frequency[carrier_index].size:
        n_freq_modes = min(result.am_frequency[carrier_index].shape[0], max_am_modes)
        max_frequency = safe_upper_frequency(result.sample_rate, max_frequency)
        fig2, ax2 = plt.subplots(figsize=(11, 3.2))
        for idx in range(n_freq_modes):
            ax2.plot(t, np.clip(result.am_frequency[carrier_index][idx], 0, max_frequency), linewidth=0.9, label=f"AM IMF {idx + 1}")
        ax2.set_ylim(0, max_frequency)
        ax2.set_xlabel("Time (s)")
        ax2.set_ylabel("AM frequency (Hz)")
        ax2.set_title(f"{label}: second-layer instantaneous AM frequency")
        ax2.legend(loc="upper right", ncol=2)
        fig2.tight_layout()

    return fig, axes


def plot_spectral_diagnostics(result, label):
    t = np.arange(result.signal.size) / result.sample_rate
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(result.carrier_bins, result.marginal, color="black", linewidth=1.0)
    axes[0].set_xscale("log")
    axes[0].set_xlabel("Carrier frequency (Hz)")
    axes[0].set_ylabel("Power")
    axes[0].set_title("Marginal HHT")

    im0 = axes[1].imshow(
        np.asarray(result.hht),
        aspect="auto",
        origin="lower",
        extent=[t[0], t[-1], result.carrier_bins[0], result.carrier_bins[-1]],
        cmap="magma",
    )
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Carrier frequency (Hz)")
    axes[1].set_title("Hilbert-Huang spectrum")
    fig.colorbar(im0, ax=axes[1], fraction=0.046, pad=0.04)

    im1 = axes[2].imshow(
        np.asarray(result.holospectrum),
        aspect="auto",
        origin="lower",
        extent=[result.am_bins[0], result.am_bins[-1], result.carrier_bins[0], result.carrier_bins[-1]],
        cmap="viridis",
    )
    axes[2].set_xscale("log")
    axes[2].set_yscale("log")
    axes[2].set_xlabel("AM frequency (Hz)")
    axes[2].set_ylabel("Carrier frequency (Hz)")
    axes[2].set_title("Time-averaged holospectrum")
    fig.colorbar(im1, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(label, y=1.03)
    fig.tight_layout()
    return fig, axes


def run_all_methods(x, sample_rate, common_kwargs, method_kwargs):
    results = {}
    failures = {}
    for label, overrides in method_kwargs.items():
        kwargs = {**common_kwargs, **overrides}
        print(f"Running {label} ...")
        try:
            result = run_hhsa(x, sample_rate, **kwargs)
            results[label] = result
            print(
                f"  extracted {result.imfs.shape[0]} IMFs; "
                f"reconstruction error={result.reconstruction_error:.3e}; "
                f"HHT={result.hht.shape}; holo={result.holospectrum.shape}"
            )
        except Exception as exc:
            failures[label] = exc
            print(f"  skipped {label}: {type(exc).__name__}: {exc}")
    print(f"Finished {len(results)} method(s), skipped {len(failures)} method(s).")
    return results, failures


def plot_method_summary(results):
    rows = []
    for label, result in results.items():
        dominant_idx = int(np.argmax(result.marginal)) if result.marginal.size else -1
        dominant_freq = result.carrier_bins[dominant_idx] if dominant_idx >= 0 else np.nan
        energies = mode_energy(result.imfs)
        rows.append((label, result.imfs.shape[0], result.reconstruction_error, dominant_freq, energies))

    for label, n_imfs, recon_error, dominant_freq, _ in rows:
        print(f"{label:14s} | IMFs={n_imfs:2d} | recon={recon_error:.3e} | dominant HHT={dominant_freq:6.2f} Hz")

    if not rows:
        return None, None

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    labels = [row[0] for row in rows]
    axes[0].bar(labels, [row[2] for row in rows], color="tab:gray")
    axes[0].set_yscale("log")
    axes[0].set_ylabel("Relative reconstruction error")
    axes[0].tick_params(axis="x", rotation=25)

    width = 0.8 / max(1, len(rows))
    for offset, (label, _, _, _, energies) in enumerate(rows):
        if energies.size == 0:
            continue
        norm_energy = energies / (np.sum(energies) + np.finfo(float).eps)
        x_pos = np.arange(norm_energy.size) + offset * width
        axes[1].bar(x_pos, norm_energy, width=width, label=label)
    axes[1].set_xlabel("IMF index")
    axes[1].set_ylabel("Normalized energy")
    axes[1].legend(loc="upper right")
    fig.tight_layout()
    return fig, axes


## Configure the Four Decomposition Methods

`run_hhsa` performs the full two-layer HHSA workflow. The `decomposition` setting selects Ensemble sift, Masked sift, CEEMDAN, or ICEEMDAN for both the carrier decomposition and the amplitude-envelope decomposition.


In [ ]:
CARRIER_LOW = 0.25
CARRIER_HIGH = safe_upper_frequency(sample_rate, 25.0)
AM_LOW = 0.1
AM_HIGH = safe_upper_frequency(sample_rate, 12.0)
MASK_FREQUENCIES_HZ = np.array([18.0, 10.0, 6.0, 2.0, 0.7])

COMMON_KWARGS = dict(
    frequency_method="hybrid",
    max_imfs=4,
    max_am_imfs=2,
    ensemble_size=8,
    noise_width=0.12,
    random_state=42,
    max_siftings=20,
    stop_sd=0.2,
    carrier_hist=(CARRIER_LOW, CARRIER_HIGH, 96, "log"),
    am_hist=(AM_LOW, AM_HIGH, 64, "log"),
    emd_backend="auto",
    sift_acceleration="none",
)

METHOD_KWARGS = {
    "Ensemble sift": dict(decomposition="ensemble_sift"),
    "Masked sift": dict(
        decomposition="mask_sift",
        mask_freqs=MASK_FREQUENCIES_HZ / sample_rate,
        mask_amp=1.0,
        mask_amp_mode="ratio_sig",
    ),
    "CEEMDAN": dict(decomposition="ceemdan"),
    "ICEEMDAN": dict(decomposition="iceemdan"),
}

print("Demo settings:")
for key, value in COMMON_KWARGS.items():
    print(f"  {key}: {value}")
print(f"  mask frequencies in Hz: {MASK_FREQUENCIES_HZ}")


## Run HHSA with Each Method

This cell catches optional backend errors and keeps the notebook running, so you can still compare whichever methods are installed in the current environment.


In [ ]:
results, failures = run_all_methods(signal, sample_rate, COMMON_KWARGS, METHOD_KWARGS)


## First-Layer Decomposition Checks

Inspect IMF ordering, mode mixing, endpoint artifacts, the slow residue, and reconstruction. The reconstruction overlay should closely match the input if decomposition completed correctly.


In [ ]:
for label, result in results.items():
    plot_decomposition_diagnostics(result, label, max_modes=COMMON_KWARGS["max_imfs"]);


## Instantaneous Amplitude and Frequency Checks

Amplitude should be non-negative and smoother than its carrier IMF. Frequency tracks should mostly sit inside plausible bands rather than being dominated by extreme spikes.


In [ ]:
for label, result in results.items():
    plot_am_fm_diagnostics(result, label, max_modes=COMMON_KWARGS["max_imfs"], max_frequency=CARRIER_HIGH);


## Second-Layer Amplitude-Modulation Checks

HHSA decomposes each carrier IMF's amplitude envelope. This section displays the strongest carrier IMF's envelope decomposition and AM-frequency tracks.


In [ ]:
for label, result in results.items():
    carrier_index = strongest_imf_index(result)
    shown = carrier_index + 1 if carrier_index is not None else "none"
    print(f"{label}: showing second-layer decomposition for carrier IMF {shown}")
    plot_second_layer_diagnostics(result, label, carrier_index=carrier_index, max_am_modes=COMMON_KWARGS["max_am_imfs"], max_frequency=AM_HIGH);


## Hilbert-Huang and Holo-Hilbert Spectral Checks

The marginal HHT summarizes carrier frequency. The HHT image shows carrier activity over time, and the holospectrum shows which amplitude-modulation frequencies are linked to each carrier band.


In [ ]:
for label, result in results.items():
    plot_spectral_diagnostics(result, label);


## Compare Methods Quantitatively

Use this compact summary to spot methods that extracted too few modes, over-split energy, or left too much structure in the residue.


In [ ]:
plot_method_summary(results);


## Direct First-Layer Calls

If you only need decomposition arrays and not the full HHSA result, call the lower-level functions directly.


In [ ]:
from hhsa import ceemdan, ensemble_sift, iceemdan, mask_sift

DIRECT_CALLS = {
    "ensemble_sift": (
        ensemble_sift,
        dict(max_imfs=4, ensemble_size=8, noise_width=0.12, random_state=42),
    ),
    "mask_sift": (
        mask_sift,
        dict(
            max_imfs=4,
            mask_freqs=MASK_FREQUENCIES_HZ / sample_rate,
            mask_amp=1.0,
            mask_amp_mode="ratio_sig",
        ),
    ),
    "ceemdan": (
        ceemdan,
        dict(max_imfs=4, ensemble_size=8, noise_width=0.12, random_state=42),
    ),
    "iceemdan": (
        iceemdan,
        dict(max_imfs=4, ensemble_size=8, noise_width=0.12, random_state=42),
    ),
}

print("Direct first-layer decomposition calls:")
for name, (function, kwargs) in DIRECT_CALLS.items():
    try:
        imfs, residue = function(signal, **kwargs)
        recon_error = np.linalg.norm(signal - (imfs.sum(axis=0) + residue)) / (np.linalg.norm(signal) + np.finfo(float).eps)
        print(f"  {name:14s}: imfs={imfs.shape}, residue={residue.shape}, recon={recon_error:.3e}")
    except Exception as exc:
        print(f"  {name:14s}: skipped ({type(exc).__name__}: {exc})")


## Practical Notes

- HHSA is applied to a one-dimensional visual trace, here ROI luminance over frames.
- For your own data, set `VIDEO_PATH` or `IMAGE_STACK_PATH` and adjust the ROI extraction in `load_visual_trace`.
- Tune `MASK_FREQUENCIES_HZ` to expected flicker, motion, stimulus, or physiological visual-response rates.
- Keep frame-rate Nyquist limits in mind; no plotted carrier or AM frequency should exceed half the frame rate.
